In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
from mnp.utils.profiler import centrar_notebook

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
centrar_notebook()

# Cargar parquet limpio
path = "/Users/abelguevarah/Desktop/invs/malnutrition-research/data/interim/v1/rec41_cleaned.parquet"
df = pd.read_parquet(path)
print(f"[OK] Cargado: {df.shape}")

2026-06-16 18:49:46.801 | INFO     | mnp.config:<module>:11 - PROJ_ROOT path is: /Users/abelguevarah/Desktop/invs/malnutrition-research


[OK] Cargado: (292824, 27)


In [2]:
# 1. Decodificación Geométrica de M34 (Cuando empezó a darle el pecho)
# 0 -> 0 horas
# 1xx -> xx horas
# 2xx -> xx días -> xx * 24 horas

def decode_m34(val):
    if pd.isna(val):
        return np.nan
    if val == 0.0:
        return 0.0
    if 100 <= val < 200:
        return val - 100
    if 200 <= val < 300:
        return (val - 200) * 24
    return np.nan # Para casos raros

df['m34_horas_inicio_lactancia'] = df['M34'].apply(decode_m34)

print("--- Auditoría de nueva variable ---")
print(df[['M34', 'm34_horas_inicio_lactancia']].value_counts(dropna=False).head(15))

--- Auditoría de nueva variable ---
M34    m34_horas_inicio_lactancia
0.0    0.0                           159030
101.0  1.0                            31464
102.0  2.0                            25103
103.0  3.0                            14782
104.0  4.0                             9535
105.0  5.0                             6867
201.0  24.0                            6067
106.0  6.0                             5247
202.0  48.0                            3707
108.0  8.0                             3389
112.0  12.0                            3075
NaN    NaN                             3022
203.0  72.0                            2958
107.0  7.0                             2556
110.0  10.0                            2003
Name: count, dtype: int64


In [3]:
# 2. Imputación Condicionada de Suplementación (M46)
# Si no tomó hierro (M45 == 0), entonces días que tomó hierro (M46) es 0.
df.loc[df['M45'] == 0, 'M46'] = 0.0

# 3. Escalamiento de Peso al Nacer (M19)
# Está en gramos (Ej. 3500.0), lo pasamos a Kilos.
df['m19_peso_nacer_kg'] = df['M19'] / 1000.0

print("Imputaciones y escalamientos completados.")

Imputaciones y escalamientos completados.


In [4]:
# 5. Agrupación Categórica de Lugar de Parto (M15)
institucional = [
    'Hospital MINSA', 'Centro de salud MINSA', 'Hospital ESSALUD',
    'Clínica privada', 'Puesto de salud MINSA', 'Consultorio médico privado',
    'Center/Posta ESSALUD', 'Hospital FF.AA / Policiales', 'Clínica de la iglesia/entidad religiosa'
]
domiciliario = [
    'Su domicilio', 'Domicilio de partera', 'Otro'
]

def clasificar_lugar(val):
    if pd.isna(val):
        return np.nan
    if val in institucional:
        return 'Institucional'
    return 'Domiciliario' # Por defecto si es otro o en casa

df['m15_lugar_parto_agrupado'] = df['M15'].apply(clasificar_lugar)

print(df['m15_lugar_parto_agrupado'].value_counts(dropna=False))

m15_lugar_parto_agrupado
Institucional    254460
Domiciliario      33715
NaN                4649
Name: count, dtype: int64


In [5]:
# 5.5 Limpieza de Variables Crudas (Early Drop)
# Eliminamos las variables originales que ya fueron transformadas a features,
# más el clon perfecto (M5 = M4) para evitar que viajen al Master.
cols_a_eliminar = ['M19', 'M34', 'M15', 'M5']
df = df.drop(columns=[c for c in cols_a_eliminar if c in df.columns])
print(f"Bajas crudas confirmadas. Columnas restantes: {df.shape[1]}")

# 6. Exportar a Features
output_path = "/Users/abelguevarah/Desktop/invs/malnutrition-research/data/interim/rec41_features.parquet"
df.to_parquet(output_path, index=False)
print(f"[OK] Guardado en: {output_path}")

Bajas crudas confirmadas. Columnas restantes: 26
[OK] Guardado en: /Users/abelguevarah/Desktop/invs/malnutrition-research/data/interim/rec41_features.parquet
